In [1]:
#%% Importing libraries
import transformers
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import re
import pandas as pd
import random
from tqdm import tqdm


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from huggingface_hub import login
login(token='')

In [92]:
#%% Load and prepare data
dataset_raw = pd.read_csv('Data/Product_Normalization_GRI.csv')
normalized_product_attributes = pd.read_excel('Data/Normalized_product_attribute_name.xlsx', sheet_name='Normalized Product Attributes')

#dataset_sample = pd.read_excel('sampled_descriptions_finetuning_llama_test.xlsx')

In [93]:
#%% Loading Preprocessed (Expanded) Dataset
dataset_raw_expanded = pd.read_csv('Data/Product_Normalization_GRI_Expanded.csv')

In [94]:
dataset_raw_expanded.head()

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...


In [95]:
dataset_raw_expanded['Guest Room Info'].unique()

array(['Accessible Room', 'Suite', 'Executive/Club Suite', 'Double Bed',
       'King Bedroom', 'Queen Bedroom', 'Penthouse', 'Studio Suite',
       'Twin Room', 'Family Room/Suite', 'Cottage', 'Loft', 'Guest Room',
       nan, 'Bungalow', 'Villa', 'Junior Suite', 'Executive/Club Room',
       'Classic Room', 'Comfort Room', 'Deluxe Room', 'Deluxe Suite',
       'Premier Room', 'Standard Room', 'Superior Room', 'Superior Suite',
       'Premier Suite', 'Luxury Room', 'Classic Suite',
       'Presidential Suite', 'Single Room', 'Studio Room', 'Cabana',
       'Apartment', 'Luxury Suite', 'Run of the House'], dtype=object)

In [96]:
# Get room types
room_types = normalized_product_attributes['RoomType'].dropna().unique()
room_types_list = room_types.tolist()
room_types_str = ", ".join(room_types_list)

In [97]:
dataset_raw_expanded['Room Description Expanded'].unique()

array(['FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO SUITE NON SMOKING VISUAL FIRE ALARM/DOOR/PHONE ALERT',
       '2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE STUDIO SUITE NON SMOKING VISUAL FIRE ALARM/DOOR/PHONE ALERT',
       'PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE STUDIO SUITE NON SMOKING VISUAL FIRE ALARM/DOOR/PHONE ALERT',
       ...,
       'BEST FLEXIBLE RATE|HOLIDAY INN EXPRESS ROOM WHEN YOU ARRIVE WE WILL DO OUR BEST TO MEET YOUR ROOM BED TYPE AND SMOKING PREFERENCES. RMS SUBJ',
       'GREATRATE DISCOUNTED STAYS. 2|RUN OF HOUSE STANDARD ROOM DOUBLE BED OR 2 SINGLES WOODEN FLOOR',
       'CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS AREA AND BREAKFAST FOR 1 SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTEL WE WILL DO OUR BEST TO MEET YOUR ROOM TYPE PREFERENCE THIS IS SUBJECT TO'],
      dtype=object)

In [125]:
def sample_descriptions_by_label(dataset, n_samples_per_class, random_state=42):
    """
    Sample Room Descriptions stratified by their Guest Room Info labels and return both sampled and remaining data
    """
    # Get unique labels
    labels = dataset['Guest Room Info'].unique()
    
    # Initialize lists to store samples
    sampled_descriptions = []
    sampled_descriptions_expanded = []  # Added for expanded descriptions
    sampled_labels = []
    sampled_indices = []
    
    print("\nSampling Room Descriptions by Label:")
    print("===================================")
    
    for label in labels:
        # Get all descriptions for this label
        label_data = dataset[dataset['Guest Room Info'] == label]
        
        # Calculate how many samples we can take
        n_available = len(label_data)
        n_to_sample = min(n_samples_per_class, n_available)
        
        if n_to_sample > 0:
            # Sample descriptions for this label
            sampled_data = label_data.sample(
                n=n_to_sample, 
                random_state=random_state
            )
            
            # Add to our lists
            sampled_descriptions.extend(sampled_data['Room Description'].tolist())
            sampled_descriptions_expanded.extend(sampled_data['Room Description Expanded'].tolist())
            sampled_labels.extend([label] * n_to_sample)
            sampled_indices.extend(sampled_data.index.tolist())
            
            print(f"\n{label}:")
            print(f"- Sampled {n_to_sample} descriptions (out of {n_available} available)")
            print(f"- Example: {sampled_data['Room Description'].iloc[0]}")
    
    # Create DataFrame of samples with both description columns
    samples_df = pd.DataFrame({
        'Room Description': sampled_descriptions,
        'Room Description Expanded': sampled_descriptions_expanded,
        'Guest Room Info': sampled_labels
    })
    
    # Create DataFrame of remaining data (not sampled)
    remaining_df = dataset[~dataset.index.isin(sampled_indices)].reset_index(drop=True)
    
    # Shuffle the samples
    samples_df = samples_df.sample(
        frac=1, 
        random_state=random_state
    ).reset_index(drop=True)
    
    print(f"\nTotal samples: {len(samples_df)} descriptions")
    print(f"Remaining data: {len(remaining_df)} descriptions")
    print("\nDistribution of labels in sample:")
    print(samples_df['Guest Room Info'].value_counts())
    
    return samples_df, remaining_df



In [126]:

# Usage:
sampled_data, remaining_data = sample_descriptions_by_label(
    dataset_raw_expanded, 
    n_samples_per_class=100
)
sampled_data.to_csv('sampled_data_finetuned_bert.csv', index = False)
remaining_data.to_csv('remaining_data_finetuned_bert.csv', index = False)


Sampling Room Descriptions by Label:

Accessible Room:
- Sampled 100 descriptions (out of 1000 available)
- Example: AAA DISCOUNT|ACCESSIBLE 2 QUEENS APPROX 395 SQ FT - JULIET BALCONY ADA ROOM - WHEELCHAIR ACCESSIBLE BATH

Suite:
- Sampled 100 descriptions (out of 1000 available)
- Example: STANDARD RATE 2BR PRES STE|2 BEDROOMS: 2 LIVING ROOMS: FREE BREAKFAST

Executive/Club Suite:
- Sampled 100 descriptions (out of 1000 available)
- Example: ROOM RATE|PREMIER FS EXECUTIVE SUITE KING BED-FLOORS 6-11 CITY AND WATER VW-SEP BEDROOM AND LIVING AREA

Double Bed:
- Sampled 100 descriptions (out of 1000 available)
- Example: FLEXIBLE RATE ROOM ONLY|DOUBLE ROOM - SUITABLE FOR 2 ADULTS

King Bedroom:
- Sampled 100 descriptions (out of 1000 available)
- Example: CCRA PGHP PROMOTIONAL RATE|BROOKLYN BRIDGE KING BED RM 300 SQ FT.

Queen Bedroom:
- Sampled 100 descriptions (out of 1000 available)
- Example: BEST AVAILABLE RATE|QUEEN ROOM W/ SOFA BED AND FRIDGE NON SMOKING FREE WI-FI/HOT BREAKFAST I

In [130]:
sampled_data.head()

,Room Description,Room Description Expanded,Guest Room Info
0,THE DINNER PACKAGE INCLUDES A|100 USD PER NIGH...,THE DINNER PACKAGE INCLUDES A|100 USD PER NIGH...,Executive/Club Room
1,SPA AND STAY|SUPERIOR SUITE-1KING OR 2TWINS-CO...,SPA AND STAY|SUPERIOR SUITE-1KING OR 2TWINS-CO...,Superior Suite
2,REFUNDABLE RATE|STANDARD KING 250SQFT.STUNNING...,REFUNDABLE RATE|STANDARD KING 250SQFT.STUNNING...,Standard Room
3,JP MORGAN CHASE|DELUXE ROOM-1KING-CITY VIEW-TV...,JP MORGAN CHASE|DELUXE ROOM-1KING-CITY VIEW-TV...,Deluxe Room
4,TRIPADVISOR PLUS|BALCONY VIEW DOUBLE DOUBLE RO...,TRIPADVISOR PLUS|BALCONY VIEW DOUBLE DOUBLE RO...,Double Bed


# Sample descriptions
n_samples_per_class = 100 # v1 -> 10, v2 -> 100
sampled_data = sample_descriptions_by_label(
    dataset_raw_expanded, 
    n_samples_per_class=n_samples_per_class
)

# Save samples (optional)
sampled_data
#.to_csv('sampled_descriptions_finetuning_llama_test.csv', index=False)

**BERT-Base-Uncased**

In [120]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import torch
from sklearn.model_selection import train_test_split

In [131]:

def train_and_evaluate_bert_base_uncased(train_data, test_data, output_dir="bert_base_uncased_checkpoint"):
    """Initial training and evaluation on split data"""
    # Get unique labels and create label mapping
    labels = train_data['Guest Room Info'].unique()
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(labels)
    
    print(f"Number of labels: {num_labels}")
    
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    def prepare_dataset(data):
        texts = data['Room Description'].tolist()
        labels = [label2id[label] for label in data['Guest Room Info']]
        
        tokenized = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        dataset = Dataset.from_dict({
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
            'labels': labels
        })
        return dataset
    
    train_dataset = prepare_dataset(train_data)
    test_dataset = prepare_dataset(test_data)
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {'accuracy': accuracy_score(labels, predictions)}
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )
    
    print("Starting initial training...")
    trainer.train()
    
    # Evaluate
    print("\nEvaluating on test set...")
    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids
    
    pred_labels_text = [id2label[id] for id in pred_labels]
    true_labels_text = [id2label[id] for id in true_labels]
    
    print("\nClassification Report:")
    print(classification_report(true_labels_text, pred_labels_text))
    
    return model_name, label2id, id2label



In [132]:
sampled_data

,Room Description,Room Description Expanded,Guest Room Info
0,THE DINNER PACKAGE INCLUDES A|100 USD PER NIGH...,THE DINNER PACKAGE INCLUDES A|100 USD PER NIGH...,Executive/Club Room
1,SPA AND STAY|SUPERIOR SUITE-1KING OR 2TWINS-CO...,SPA AND STAY|SUPERIOR SUITE-1KING OR 2TWINS-CO...,Superior Suite
2,REFUNDABLE RATE|STANDARD KING 250SQFT.STUNNING...,REFUNDABLE RATE|STANDARD KING 250SQFT.STUNNING...,Standard Room
3,JP MORGAN CHASE|DELUXE ROOM-1KING-CITY VIEW-TV...,JP MORGAN CHASE|DELUXE ROOM-1KING-CITY VIEW-TV...,Deluxe Room
4,TRIPADVISOR PLUS|BALCONY VIEW DOUBLE DOUBLE RO...,TRIPADVISOR PLUS|BALCONY VIEW DOUBLE DOUBLE RO...,Double Bed
...,...,...,...
3495,UNLIMITED SINGLE GOLF EXPERIEN|OUR 1811 KING C...,UNLIMITED SINGLE GOLF EXPERIEN|OUR 1811 KING C...,Cottage
3496,15PCT OFF. BKFST ART SUITE|KING BED:120SQM:LOF...,15PCT OFF. BKFST ART SUITE|KING BED:120SQM:LOF...,Loft
3497,PREPAY NONREF BKFT|PREPAY NON-REFUNDABLE WITH ...,PREPAY NONREF BKFT|PREPAY NON-REFUNDABLE WITH ...,Guest Room
3498,ACCENTURE-BREAKFAST INCLUDED|SUPERIOR DOUBLE O...,ACCENTURE-BREAKFAST INCLUDED|SUPERIOR DOUBLE O...,Twin Room


In [133]:
def train_final_model(full_sample_data, model_name, label2id, output_dir="final_bert_base_uncased"):
    """Train final model on all 350 or 3500 samples"""
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(label2id)
    
    print("\nTraining final model on all sample data...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    # Prepare full dataset
    texts = full_sample_data['Room Description'].tolist()
    labels = [label2id[label] for label in full_sample_data['Guest Room Info']]
    
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    full_dataset = Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': labels
    })
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,  # More epochs for final training
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        save_strategy="epoch"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_dataset,
        tokenizer=tokenizer
    )
    
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    return model, tokenizer


In [139]:
def classify_remaining_data(model, tokenizer, label2id, remaining_data):
    """Classify the remaining 35k descriptions"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = model.to(device)
    
    id2label = {i: label for label, i in label2id.items()}
    results = []
    
    print("\nClassifying remaining descriptions...")
    for _, row in tqdm(remaining_data.iterrows(), total=len(remaining_data)):
        inputs = tokenizer(
            row['Room Description'],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_id = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_id].item()
        
        results.append({
            'Room Description': row['Room Description'],
            'Actual_Room_Type': row['Guest Room Info'],  # Added actual label
            'Predicted_Room_Type': id2label[predicted_id],
            'Confidence': confidence
        })
    
    results_df = pd.DataFrame(results)
    
    # Add accuracy column
    results_df['Correct'] = results_df['Actual_Room_Type'] == results_df['Predicted_Room_Type']
    
    # Print overall accuracy
    accuracy = results_df['Correct'].mean()
    print(f"\nOverall Accuracy: {accuracy:.2%}")
    
    return results_df

In [83]:
sampled_data

,Room Description,Guest Room Info
0,THE DINNER PACKAGE INCLUDES A|100 USD PER NIGH...,Executive/Club Room
1,SPA AND STAY|SUPERIOR SUITE-1KING OR 2TWINS-CO...,Superior Suite
2,REFUNDABLE RATE|STANDARD KING 250SQFT.STUNNING...,Standard Room
3,JP MORGAN CHASE|DELUXE ROOM-1KING-CITY VIEW-TV...,Deluxe Room
4,TRIPADVISOR PLUS|BALCONY VIEW DOUBLE DOUBLE RO...,Double Bed
...,...,...
3495,UNLIMITED SINGLE GOLF EXPERIEN|OUR 1811 KING C...,Cottage
3496,15PCT OFF. BKFST ART SUITE|KING BED:120SQM:LOF...,Loft
3497,PREPAY NONREF BKFT|PREPAY NON-REFUNDABLE WITH ...,Guest Room
3498,ACCENTURE-BREAKFAST INCLUDED|SUPERIOR DOUBLE O...,Twin Room


In [135]:
# Main execution
# 1. Split sample data and validate
train_data, test_data = train_test_split(
    sampled_data, 
    test_size=0.2, 
    stratify=sampled_data['Guest Room Info'],
    random_state=42
)

# Initial training and evaluation
model_name, label2id, id2label = train_and_evaluate_bert_base_uncased(train_data, test_data)

Number of labels: 35


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_26682/10981553.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting initial training...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.447007,0.711429
2,No log,1.365072,0.894286
3,2.198400,1.080051,0.921429


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Evaluating on test set...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Classification Report:
                      precision    recall  f1-score   support

     Accessible Room       0.89      0.80      0.84        20
           Apartment       0.95      1.00      0.98        20
            Bungalow       1.00      1.00      1.00        20
              Cabana       1.00      1.00      1.00        20
        Classic Room       0.86      0.95      0.90        20
       Classic Suite       0.94      0.85      0.89        20
        Comfort Room       1.00      1.00      1.00        20
             Cottage       1.00      1.00      1.00        20
         Deluxe Room       0.90      0.90      0.90        20
        Deluxe Suite       0.95      0.90      0.92        20
          Double Bed       0.95      0.90      0.92        20
 Executive/Club Room       0.90      0.90      0.90        20
Executive/Club Suite       0.95      1.00      0.98        20
   Family Room/Suite       1.00      0.95      0.97        20
          Guest Room       0.95      0.90    

In [136]:
# 2. Train on full sample dataset
final_model, final_tokenizer = train_final_model(sampled_data, model_name, label2id)



Training final model on all sample data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_26682/1263641235.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,2.115700
1000,0.377700


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/s

In [138]:
dataset_raw_expanded

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...
...,...,...,...,...
34995,34995,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...
34996,34996,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...
34997,34997,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...
34998,34998,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO..."


dataset_raw_expanded[~dataset_raw_expanded.index.isin(sampled_data.index)]

In [140]:
# 3. Classify remaining data
#remaining_data = remaining_data
results_df = classify_remaining_data(final_model, final_tokenizer, label2id, remaining_data)



Classifying remaining descriptions...


100%|██████████| 31500/31500 [11:45<00:00, 44.63it/s]



Overall Accuracy: 95.73%


In [108]:
remaining_data

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
3500,3500,MRATE AVAIL BOOK WITH CONFIDEN|PRIVILEGE DOUBL...,Double Bed,MRATE AVAIL BOOK WITH CONFIDEN|PRIVILEGE DOUBL...
3501,3501,FLEXIBLE - RATE|SUPERIOR DOUBLE ROOM,Double Bed,FLEXIBLE - RATE|SUPERIOR DOUBLE ROOM
3502,3502,TRIP PLUS BED AND BREAKFAST|DOUBLE ROOM WITH B...,Double Bed,TRIP PLUS BED AND BREAKFAST|DOUBLE ROOM WITH B...
3503,3503,ACCENTURE|DOUBLE ROOM - 24SQM - 258SQFT - KING...,Double Bed,ACCENTURE|DOUBLE ROOM - 24SQM - 258SQFT - KING...
3504,3504,FLEXIBLE - RATE WITH BREAKFAST|SUPERIOR DOUBLE...,Double Bed,FLEXIBLE - RATE WITH BREAKFAST|SUPERIOR DOUBLE...
...,...,...,...,...
34995,34995,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...
34996,34996,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...
34997,34997,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...
34998,34998,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO..."


In [47]:
remaining_data_reset = remaining_data.reset_index(drop=True)
remaining_data_reset

,Room Description,Guest Room Info
0,MRATE AVAIL BOOK WITH CONFIDEN|PRIVILEGE DOUBL...,Double Bed
1,FLEXIBLE - RATE|SUPERIOR DOUBLE ROOM,Double Bed
2,TRIP PLUS BED AND BREAKFAST|DOUBLE ROOM WITH B...,Double Bed
3,ACCENTURE|DOUBLE ROOM - 24SQM - 258SQFT - KING...,Double Bed
4,FLEXIBLE - RATE WITH BREAKFAST|SUPERIOR DOUBLE...,Double Bed
...,...,...
31495,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House
31496,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House
31497,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House
31498,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House


In [109]:
results_df

,Room Description,Actual_Room_Type,Predicted_Room_Type,Confidence,Correct
0,MRATE AVAIL BOOK WITH CONFIDEN|PRIVILEGE DOUBL...,Double Bed,Double Bed,0.854171,True
1,FLEXIBLE - RATE|SUPERIOR DOUBLE ROOM,Double Bed,Double Bed,0.847689,True
2,TRIP PLUS BED AND BREAKFAST|DOUBLE ROOM WITH B...,Double Bed,Double Bed,0.868048,True
3,ACCENTURE|DOUBLE ROOM - 24SQM - 258SQFT - KING...,Double Bed,Double Bed,0.870517,True
4,FLEXIBLE - RATE WITH BREAKFAST|SUPERIOR DOUBLE...,Double Bed,Double Bed,0.845350,True
...,...,...,...,...,...
31495,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House,Run of the House,0.876735,True
31496,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House,Run of the House,0.905424,True
31497,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House,Run of the House,0.914606,True
31498,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House,Run of the House,0.837103,True


In [88]:
'''
# Add actual labels to existing results
results_df['Actual_Room_Type'] = remaining_data_reset['Guest Room Info']

# Add accuracy column
results_df['Correct'] = results_df['Actual_Room_Type'] == results_df['Predicted_Room_Type']

# Print overall accuracy
accuracy = results_df['Correct'].mean()
print(f"\nOverall Accuracy: {accuracy:.2%}")
'''
# Save updated results
results_df #.to_csv('classified_rooms_bert_base_uncased_v3.csv', index=False)

,Room Description,Actual_Room_Type,Predicted_Room_Type,Confidence,Correct
0,MRATE AVAIL BOOK WITH CONFIDEN|PRIVILEGE DOUBL...,Double Bed,Double Bed,0.065499,True
1,FLEXIBLE - RATE|SUPERIOR DOUBLE ROOM,Double Bed,Double Bed,0.062171,True
2,TRIP PLUS BED AND BREAKFAST|DOUBLE ROOM WITH B...,Double Bed,Twin Room,0.065648,False
3,ACCENTURE|DOUBLE ROOM - 24SQM - 258SQFT - KING...,Double Bed,Single Room,0.062879,False
4,FLEXIBLE - RATE WITH BREAKFAST|SUPERIOR DOUBLE...,Double Bed,Double Bed,0.064722,True
...,...,...,...,...,...
31495,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House,Run of the House,0.108942,True
31496,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House,Run of the House,0.112920,True
31497,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House,Run of the House,0.111424,True
31498,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House,Run of the House,0.101761,True


In [110]:
# Save results
results_df.to_csv('classified_rooms_bert_base_uncased_v3.csv', index=False)
print("\nClassification complete! Results saved to 'classified_rooms_bert_base_uncased_v3.csv'")


Classification complete! Results saved to 'classified_rooms_bert_base_uncased_v3.csv'


**MiniLM-L12-H384-uncased**

In [29]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import torch
from sklearn.model_selection import train_test_split

In [30]:

def train_and_evaluate_minilm(train_data, test_data, output_dir="minilm_checkpoint"):
    """Initial training and evaluation on split data"""
    # Get unique labels and create label mapping
    labels = train_data['Guest Room Info'].unique()
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(labels)
    
    print(f"Number of labels: {num_labels}")
    
    model_name = "microsoft/MiniLM-L12-H384-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    def prepare_dataset(data):
        texts = data['Room Description'].tolist()
        labels = [label2id[label] for label in data['Guest Room Info']]
        
        tokenized = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        dataset = Dataset.from_dict({
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
            'labels': labels
        })
        return dataset
    
    train_dataset = prepare_dataset(train_data)
    test_dataset = prepare_dataset(test_data)
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {'accuracy': accuracy_score(labels, predictions)}
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )
    
    print("Starting initial training...")
    trainer.train()
    
    # Evaluate
    print("\nEvaluating on test set...")
    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids
    
    pred_labels_text = [id2label[id] for id in pred_labels]
    true_labels_text = [id2label[id] for id in true_labels]
    
    print("\nClassification Report:")
    print(classification_report(true_labels_text, pred_labels_text))
    
    return model_name, label2id, id2label



In [31]:
def train_final_model_minilm(full_sample_data, model_name, label2id, output_dir="final_minilm"):
    """Train final model on all 350 samples"""
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(label2id)
    
    print("\nTraining final model on all sample data...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    # Prepare full dataset
    texts = full_sample_data['Room Description'].tolist()
    labels = [label2id[label] for label in full_sample_data['Guest Room Info']]
    
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    full_dataset = Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': labels
    })
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,  # More epochs for final training
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        save_strategy="epoch"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_dataset,
        tokenizer=tokenizer
    )
    
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    return model, tokenizer




In [32]:
def classify_remaining_data_minilm(model, tokenizer, label2id, remaining_data):
    """Classify the remaining 35k descriptions"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = model.to(device)
    
    id2label = {i: label for label, i in label2id.items()}
    results = []
    
    print("\nClassifying remaining descriptions...")
    for _, row in tqdm(remaining_data.iterrows(), total=len(remaining_data)):
        inputs = tokenizer(
            row['Room Description'],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_id = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_id].item()
        
        results.append({
            'Room Description': row['Room Description'],
            'Actual_Room_Type': row['Guest Room Info'],  # Added actual label
            'Predicted_Room_Type': id2label[predicted_id],
            'Confidence': confidence
        })
    
    results_df = pd.DataFrame(results)
    
    # Add accuracy column
    results_df['Correct'] = results_df['Actual_Room_Type'] == results_df['Predicted_Room_Type']
    
    # Print overall accuracy
    accuracy = results_df['Correct'].mean()
    print(f"\nOverall Accuracy: {accuracy:.2%}")
    
    return results_df

In [51]:
# Main execution
# 1. Split sample data and validate
train_data, test_data = train_test_split(
    sampled_data, 
    test_size=0.2, 
    stratify=sampled_data['Guest Room Info'],
    random_state=42
)

# Initial training and evaluation
model_name, label2id, id2label = train_and_evaluate_minilm(train_data, test_data)

Number of labels: 35


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_26682/248026038.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting initial training...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,3.413036,0.068571
2,No log,3.232183,0.130000
3,3.352500,3.174551,0.152857


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Evaluating on test set...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Classification Report:
                      precision    recall  f1-score   support

     Accessible Room       0.00      0.00      0.00        20
           Apartment       0.00      0.00      0.00        20
            Bungalow       0.14      1.00      0.25        20
              Cabana       0.00      0.00      0.00        20
        Classic Room       0.00      0.00      0.00        20
       Classic Suite       0.12      0.20      0.15        20
        Comfort Room       0.00      0.00      0.00        20
             Cottage       0.00      0.00      0.00        20
         Deluxe Room       0.88      0.70      0.78        20
        Deluxe Suite       0.00      0.00      0.00        20
          Double Bed       0.00      0.00      0.00        20
 Executive/Club Room       0.00      0.00      0.00        20
Executive/Club Suite       0.27      0.30      0.29        20
   Family Room/Suite       0.00      0.00      0.00        20
          Guest Room       0.23      0.75    

/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

In [85]:

# 2. Train on full sample dataset
final_model, final_tokenizer = train_final_model_minilm(sampled_data, model_name, label2id)



Training final model on all sample data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_26682/3187214905.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,2.065200
1000,0.339400


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/s

In [86]:
final_model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [87]:

# 3. Classify remaining data
remaining_data = dataset_raw[~dataset_raw.index.isin(sampled_data.index)]
results_df = classify_remaining_data_minilm(final_model, final_tokenizer, label2id, remaining_data)


Classifying remaining descriptions...


 89%|████████▉ | 28174/31500 [36:32<04:18, 12.85it/s]  


KeyboardInterrupt: 

In [57]:
# Save results
results_df.to_csv('classified_rooms_minilm_v2.csv', index=False)
print("\nClassification complete! Results saved to 'classified_rooms_minilm_v2.csv'")


Classification complete! Results saved to 'classified_rooms_minilm_v2.csv'
